# 讓 Agent 接得住前後文這份教材會先準備對話識別碼，再建立流程，最後用同一個識別碼跑多輪。這樣你可以清楚看到前後文怎麼被保存。

## 在 Colab 準備環境先準備 SDK 執行環境。

In [ ]:
from pathlib import Pathimport os, sys, subprocessif not Path('agentic_sdk').exists():    if not Path('Agentic-SDK').exists():        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)    os.chdir('Agentic-SDK')subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)print('Agentic SDK ready')

## 載入流程元件多輪對話不需要先換很多模組。重點是同一條流程要用同一個對話識別碼。

In [ ]:
from agentic_sdk import Workflowfrom agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 先準備對話識別碼同一個使用者、同一段對話，應該使用同一個 `session_id`。換掉它，就等於開一段新對話。

In [ ]:
session_id = 'demo-user-001'session_id

## 準備這段對話會用到的資料這裡把專案代號和會議資訊做成可查資料，讓後面兩輪對話有內容可以承接。

In [ ]:
knowledge_items = [    {'keywords': ['專案代號', 'aurora'], 'content': '使用者提到的專案代號是 Aurora。'},    {'keywords': ['會議', '明天'], 'content': '明天會議需要準備專案摘要。'},]

## 分別宣告模組先把輸入整理、查資料、回答三個角色分開建立，等一下再接成完整流程。

In [ ]:
perceive = PassThroughPerceive()retrieve = KeywordRetrieve(items=knowledge_items)action = DirectAnswerAction()

## 建立可以多輪使用的流程工作流程本身可以重複使用；是否接得住前後文，取決於執行時使用的 `session_id`。

In [ ]:
workflow = Workflow(    workflow_name='多輪問答 Agent',    perceive=perceive,    retrieve=retrieve,    action=action,)

## 第一輪：先告訴 Agent 一件事第一輪會把使用者訊息和 Agent 回覆保存到這個 `session_id` 底下。

In [ ]:
first = workflow.run('請記住，這次專案代號是 Aurora。', session_id=session_id)print(first.final_message)

## 第二輪：接著問後續問題第二輪仍然使用同一個 `session_id`，所以記憶裡會保留前一輪內容。

In [ ]:
second = workflow.run('那明天會議我要準備什麼？', session_id=session_id)print(second.final_message)

## 查看保存下來的對話紀錄這裡看的是記憶類型保存的完整 turn 順序，不是單次執行的中間資料。

In [ ]:
memory = second.memoryfor index, turn in enumerate(memory.turns, start=1):    print(index, turn.role, '=>', turn.content)

## 換一個對話識別碼看看換成新的 `session_id`，就會得到一段新的對話記憶。這可以避免不同使用者或不同任務互相污染。

In [ ]:
new_session = workflow.run('剛剛的專案代號是什麼？', session_id='another-session')print(new_session.final_message)print('turns:', len(new_session.memory.turns))